In [1]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

df_aggregator = pd.read_csv('../data/raw/chainlink_aggregator_raw.csv')
df_aggregator.head()

,log_index,transaction_hash,transaction_index,address,data,topic0,topic1,topic2,topic3,block_timestamp,block_number,block_hash
0,179,0x9b8eb8418984c63674b0646afe7feb5ffa8ff5cef3ed...,149,0xbd11bc57fc140614190cabc1b4c316aba220bae4,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:35:54+00:00,12276222,0xaad0334e922f98342d3294eb046abbe9b13f660fe8e9...
1,270,0xa24af7eb104a9b055b702a47fffb165c4037a5ff3b08...,257,0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:35:22+00:00,12276220,0x6bfbc354c382fc9361c6aedf552c37591a22591fff9b...
2,107,0x9e2912eaae8aa81f2a6f9353b2a6905d35552c0eff09...,90,0xbd11bc57fc140614190cabc1b4c316aba220bae4,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:31:22+00:00,12276208,0x3844d80a3e17bd1cc711b1380f592059222c347b451d...
3,105,0x29e1941085315edf8c8e9dff8f402b7f7803e76a5f41...,177,0x7104ac4abcecf1680f933b04c214b0c491d43ecc,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:00:21+00:00,12276061,0x24eab022b4b871476fc02be999aeefa37bb787043a9f...
4,91,0x8a8df95ec5ad887712c7ac8db6934daae103a1bdc368...,113,0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 08:59:58+00:00,12276059,0x37751db45d4149a4f3a73190faa241f0de845e433961...


In [2]:
chainlink_aggregator_address_mapping = {
    # BTCUSD
    '0xf5fff180082d6017036b771ba883025c654bc935': 'BTCUSD',
    '0x276de5c241071b4728697f4e11a377484a2dd6cb': 'BTCUSD',
    '0xf570deefff684d964dc3e15e1f9414283e3f7419': 'BTCUSD',
    '0x7104ac4abcecf1680f933b04c214b0c491d43ecc': 'BTCUSD',
    '0xae74faa92cb67a95ebcab07358bc222e33a34da7': 'BTCUSD',
    '0xdbe1941bfbe4410d6865b9b7078e0b49af144d2d': 'BTCUSD',
    '0x4a3411ac2948b33c69666b35cc6d055b27ea84f1': 'BTCUSD',
    # ETHUSD
    '0xf79d6afbb6da890132f9d7c355e3015f15f3406f': 'ETHUSD',
    '0xb103ede8acd6f0c106b7a5772e9d24e34f5ebc2c': 'ETHUSD',
    '0x00c7a37b03690fb9f41b5c5af8131735c7275446': 'ETHUSD',
    '0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd': 'ETHUSD',
    '0x37bc7498f4ff12c19678ee8fe19d713b87f6a9e6': 'ETHUSD',
    '0xe62b71cf983019bff55bc83b48601ce8419650cc': 'ETHUSD',
    # LINKUSD
    '0x32dbd3214ac75223e27e575c53944307914f7a90': 'LINKUSD',
    '0xc8b5381a98c7dc8b91f4149303397da56061ebaf': 'LINKUSD',
    '0x8cde021f0bfa5f82610e8ce46493cf66ac04af53': 'LINKUSD',
    '0xbd11bc57fc140614190cabc1b4c316aba220bae4': 'LINKUSD',
    '0xdfd03bfc3465107ce570a0397b247f546a42d0fa': 'LINKUSD',
    '0x20807cf61ad17c31837776fa39847a2fa1839e81': 'LINKUSD',
    '0x96d6e33b411dc1f4e3f1e894a5a5d9ce0f96738d': 'LINKUSD',
    # USDCUSD
    '0x3b15a92872435c01c27201aae0968839fb45217d': 'USDCUSD',
    '0x789190466e21a8b78b8027866cbbdc151542a26c': 'USDCUSD',
    '0xc9e1a09622afdb659913fefe800feae5dbbfe9d7': 'USDCUSD',
}

In [3]:
from helper import hex_to_uint, hex_to_int_signed_256, first_data_word, split_abi_words

In [4]:
df_aggregator['block_timestamp'] = pd.to_datetime(df_aggregator["block_timestamp"], errors="coerce", utc=True)
df_aggregator["date"] = df_aggregator["block_timestamp"].dt.date
df_aggregator["minute"] = df_aggregator["block_timestamp"].dt.floor("min")

df_aggregator['record_type'] = 'chainlink_oracle_update'
df_aggregator['feed_name'] = df_aggregator['address'].map(chainlink_aggregator_address_mapping)

# Decode Chainlink fields
df_aggregator["oracle_price_raw"] = df_aggregator["topic1"].apply(hex_to_int_signed_256)
df_aggregator["oracle_price"] = df_aggregator["oracle_price_raw"] / 1e8

df_aggregator["round_id"] = df_aggregator["topic2"].apply(hex_to_uint)

df_aggregator["updated_at_raw_hex"] = df_aggregator["data"].apply(first_data_word)
df_aggregator["updated_at_unix"] = df_aggregator["updated_at_raw_hex"].apply(hex_to_uint)
df_aggregator["updated_at_datetime"] = pd.to_datetime(
    df_aggregator["updated_at_unix"],
    unit="s",
    errors="coerce",
    utc=True
)

In [5]:
word_lists = df_aggregator["data"].apply(split_abi_words)

max_words = word_lists.apply(len).max()

# Create decoded columns
for i in range(int(max_words)):
    df_aggregator[f"data_word_{i}"] = word_lists.apply(lambda w: w[i] if len(w) > i else np.nan)
    df_aggregator[f"data_word_{i}_uint"] = df_aggregator[f"data_word_{i}"].apply(hex_to_uint)
    df_aggregator[f"data_word_{i}_int"] = df_aggregator[f"data_word_{i}"].apply(hex_to_int_signed_256)

In [6]:
df_aggregator.head()

,log_index,transaction_hash,transaction_index,address,data,topic0,topic1,topic2,topic3,block_timestamp,block_number,block_hash,date,minute,record_type,feed_name,oracle_price_raw,oracle_price,round_id,updated_at_raw_hex,updated_at_unix,updated_at_datetime,data_word_0,data_word_0_uint,data_word_0_int
0,179,0x9b8eb8418984c63674b0646afe7feb5ffa8ff5cef3ed...,149,0xbd11bc57fc140614190cabc1b4c316aba220bae4,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:35:54+00:00,12276222,0xaad0334e922f98342d3294eb046abbe9b13f660fe8e9...,2021-04-20,2021-04-20 09:35:00+00:00,chainlink_oracle_update,LINKUSD,3670064564,36.700646,886,0x00000000000000000000000000000000000000000000...,1618911354,2021-04-20 09:35:54+00:00,0x00000000000000000000000000000000000000000000...,1618911354,1618911354
1,270,0xa24af7eb104a9b055b702a47fffb165c4037a5ff3b08...,257,0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:35:22+00:00,12276220,0x6bfbc354c382fc9361c6aedf552c37591a22591fff9b...,2021-04-20,2021-04-20 09:35:00+00:00,chainlink_oracle_update,ETHUSD,215536000000,2155.360000,1763,0x00000000000000000000000000000000000000000000...,1618911322,2021-04-20 09:35:22+00:00,0x00000000000000000000000000000000000000000000...,1618911322,1618911322
2,107,0x9e2912eaae8aa81f2a6f9353b2a6905d35552c0eff09...,90,0xbd11bc57fc140614190cabc1b4c316aba220bae4,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:31:22+00:00,12276208,0x3844d80a3e17bd1cc711b1380f592059222c347b451d...,2021-04-20,2021-04-20 09:31:00+00:00,chainlink_oracle_update,LINKUSD,3629919496,36.299195,885,0x00000000000000000000000000000000000000000000...,1618911082,2021-04-20 09:31:22+00:00,0x00000000000000000000000000000000000000000000...,1618911082,1618911082
3,105,0x29e1941085315edf8c8e9dff8f402b7f7803e76a5f41...,177,0x7104ac4abcecf1680f933b04c214b0c491d43ecc,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 09:00:21+00:00,12276061,0x24eab022b4b871476fc02be999aeefa37bb787043a9f...,2021-04-20,2021-04-20 09:00:00+00:00,chainlink_oracle_update,BTCUSD,5526360967235,55263.609672,1473,0x00000000000000000000000000000000000000000000...,1618909221,2021-04-20 09:00:21+00:00,0x00000000000000000000000000000000000000000000...,1618909221,1618909221
4,91,0x8a8df95ec5ad887712c7ac8db6934daae103a1bdc368...,113,0xd3fcd40153e56110e6eeae13e12530e26c9cb4fd,0x00000000000000000000000000000000000000000000...,0x0559884fd3a460db3073b7fc896cc77986f16e378210...,0x00000000000000000000000000000000000000000000...,0x00000000000000000000000000000000000000000000...,NaN,2021-04-20 08:59:58+00:00,12276059,0x37751db45d4149a4f3a73190faa241f0de845e433961...,2021-04-20,2021-04-20 08:59:00+00:00,chainlink_oracle_update,ETHUSD,214432000000,2144.320000,1762,0x00000000000000000000000000000000000000000000...,1618909198,2021-04-20 08:59:58+00:00,0x00000000000000000000000000000000000000000000...,1618909198,1618909198


In [7]:
df_aggregator.to_csv('../data/processed/chainlink_aggregator_processed.csv', index=False)